# 🚂 TRAIN models IU-CXR — `model_og_IU` + `model_retrained_iu_3per` (EXP7-I)

> 🎯 **Mục đích**: Train 2 model bắt buộc cho unlearning trên Indiana University CXR:
> - **`model_og_IU`** — train trên TOÀN BỘ IU train fold (model gốc cần unlearn).
> - **`model_retrained_iu_3per`** — train trên IU train fold TRỪ forget 3% (gold standard cho metric 1−CosSim).
>
> Output → đóng gói thành Kaggle Dataset **`forget-mi-models-iu`** để chạy baseline + LoKU.

## ⚠️ Phụ thuộc (phải có TRƯỚC khi chạy)
| Input | Để làm gì |
|---|---|
| Kaggle Dataset **`forget-mi-data-iu`** | metadata + img_data + splits (tạo từ `preprocess_iu_kaggle.ipynb`) |
| Kaggle Dataset **`forget-mi-models`** (hoặc `forget-mi-models-full`) | Lấy model MIMIC làm **warm-start** (`--init_from`) — cùng tokenizer vocab 31090 |
| **GPU** T4/P100 + Internet ON | Train + git clone |

> 💡 **Warm-start là cố ý**: cả 2 dataset đều là chest X-ray + report lâm sàng → transfer learning chest-xray→chest-xray, hội tụ nhanh trong giới hạn 12h của Kaggle. (Train from scratch BERT+ResNet trên ~3K mẫu sẽ không kịp/không hội tụ tốt.) Ghi rõ điều này trong luận văn.

## Thứ tự chạy
`Cell 1` (setup) → `Cell 2` (detect + verify) → `Cell 3` (train og, ~8–14h) → `Cell 4` (train re 3%, ~8–14h) → `Cell 5` (verify) → **Create Dataset from Output** = `forget-mi-models-iu`

> ⏱ **Quota**: mỗi model ~8–14h. Nếu free 30h/tuần, có thể tách Cell 3 và Cell 4 ra 2 session. Cell 2 idempotent, Cell 4 độc lập với Cell 3 (đều warm-start từ MIMIC).


In [ ]:
# ====================================
# CELL 1: Setup Kaggle env (clone repo + install deps)
# ====================================
import os, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Forget-MI-LoKU"

def _get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return None

_token = _get_token()
url = f"https://{_token}@github.com/nhnhu146/Forget-MI-LoKU.git" if _token else "https://github.com/nhnhu146/Forget-MI-LoKU.git"

os.chdir(WORK)
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
else:
    if _token:
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)
os.chdir(REPO_DIR)
print("📂 CWD:", os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-1'])

print("\n📦 Installing deps...")
get_ipython().system('pip install -q pydicom scikit-image pyyaml')
get_ipython().system('pip install -q --force-reinstall --no-deps "transformers==4.38.0" "tokenizers==0.15.2" "peft==0.10.0" "accelerate==0.27.0"')

import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"\n🟢 GPU: {p.name} ({p.total_memory/1e9:.1f} GB)")
else:
    print("\n🔴 KHÔNG CÓ GPU — Settings → Accelerator → GPU T4 x2")
print("✅ Setup complete")

In [ ]:
# ====================================
# CELL 2: Auto-detect IU data + MIMIC init model + GPU (recursive — robust với mọi nesting)
# ====================================
import glob, os

def _find_one(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    return None

# --- IU data (từ forget-mi-data-iu) ---
_tsv      = _find_one(['/kaggle/input/**/all_data.tsv'])
IU_META   = os.path.dirname(_tsv) if _tsv else None
IU_IMG    = None
if IU_META:
    sib = os.path.join(os.path.dirname(IU_META), 'img_data')
    IU_IMG = sib if os.path.isdir(sib) else _find_one(['/kaggle/input/**/img_data'])
    # Cách 3: forget-mi-data-iu KHÔNG chứa ảnh → trỏ thẳng sang dataset raddar (phải attach kèm).
    if not IU_IMG:
        for _r in glob.glob('/kaggle/input/*chest-xray*') + glob.glob('/kaggle/input/datasets/*/*chest-xray*'):
            for _sub in ('images/images_normalized', 'images/images', 'images_normalized', 'images'):
                _p = os.path.join(_r, _sub)
                if os.path.isdir(_p) and glob.glob(os.path.join(_p, '*.png')):
                    IU_IMG = _p; break
            if IU_IMG: break
IU_SPLIT   = _find_one(['/kaggle/input/**/iu-split.csv'])
IU_FORGET3 = _find_one(['/kaggle/input/**/forget_set_3per_iu.csv'])

# --- MIMIC base model để warm-start (--init_from) ---
_init_bin = _find_one([
    '/kaggle/input/**/original_model/forgetme/training_original_model/pytorch_model.bin',
    '/kaggle/input/**/base_model/training_original_model/pytorch_model.bin',
    '/kaggle/input/**/training_original_model/pytorch_model.bin',
])
INIT_DIR = os.path.dirname(_init_bin) if _init_bin else None

print("🔍 Detected paths:")
for name, val in [("IU metadata (all_data.tsv)", IU_META), ("IU img_data", IU_IMG),
                  ("IU split (iu-split.csv)", IU_SPLIT), ("IU forget 3%", IU_FORGET3),
                  ("MIMIC init model (warm-start)", INIT_DIR)]:
    print(f"  {'✅' if val else '❌'} {name:<32} {val}")

import torch
print(f"\n  {'🟢' if torch.cuda.is_available() else '🔴'} GPU available: {torch.cuda.is_available()}")

ready = all([IU_META, IU_IMG, IU_SPLIT, IU_FORGET3, INIT_DIR])
print("\n" + ("✅ READY — chạy Cell 3 + Cell 4" if ready else
      "❌ THIẾU input — Add Data: forget-mi-data-iu + forget-mi-models (cho warm-start)"))

In [ ]:
# ====================================
# CELL: Train model_og_IU  — cờ RUN_OG + resume checkpoint (chống Kaggle 12h)
# ====================================
import os, subprocess, glob, json

OG_OUT = "/kaggle/working/forget-mi-models-iu/base_model/model_og_IU"
EPOCHS = 20          # ⚙️ giảm nếu sợ vượt 12h; 15–20 đủ với warm-start
LR, BS = "2e-5", "16"
RUN_OG = True        # ⚙️ True/False để chọn train model này

def _find_resume(name):
    # Tìm checkpoint từ output dataset của run TRƯỚC (attach vào Input).
    hits = glob.glob(f'/kaggle/input/**/{name}/_train_state.json', recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if not RUN_OG:
    print("⏭️  Skip model_og_IU (RUN_OG=False)")
else:
    _resume = _find_resume("model_og_IU")
    _done = False
    if _resume:
        _st = json.load(open(os.path.join(_resume, "_train_state.json")))
        if int(_st.get("epoch", 0)) >= EPOCHS:
            print(f"✅ model_og_IU đã train đủ {EPOCHS} epoch (từ {_resume}) — skip")
            _done = True
        else:
            print(f"♻️  Sẽ resume từ epoch {_st.get('epoch')} ({_resume})")
    if not _done:
        cmd = [
            "python", "scripts/train_iu_model.py",
            "--config", "config_baseline_iu_kaggle.yaml",
            "--mode", "og",
            "--init_from", INIT_DIR,
            "--text_data_dir", IU_META, "--img_data_dir", IU_IMG, "--data_split", IU_SPLIT,
            "--output_dir", OG_OUT,
            "--epochs", str(EPOCHS), "--lr", LR, "--batch_size", BS, "--seed", "42",
        ]
        if _resume:
            cmd += ["--resume_from", _resume]
        print("▶", " ".join(cmd))
        rc = subprocess.run(cmd, env=dict(os.environ, PYTHONPATH=".")).returncode
        if rc != 0:
            raise RuntimeError(f"train model_og_IU failed (exit {rc})")


In [ ]:
# ====================================
# CELL: Train model_retrained_iu_3per  — cờ RUN_RE + resume checkpoint (chống Kaggle 12h)
# ====================================
import os, subprocess, glob, json

RE_OUT = "/kaggle/working/forget-mi-models-iu/retrained_model/model_retrained_iu_3per"
EPOCHS = 20          # ⚙️ giảm nếu sợ vượt 12h; 15–20 đủ với warm-start
LR, BS = "2e-5", "16"
RUN_RE = True        # ⚙️ True/False để chọn train model này

def _find_resume(name):
    # Tìm checkpoint từ output dataset của run TRƯỚC (attach vào Input).
    hits = glob.glob(f'/kaggle/input/**/{name}/_train_state.json', recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if not RUN_RE:
    print("⏭️  Skip model_retrained_iu_3per (RUN_RE=False)")
else:
    _resume = _find_resume("model_retrained_iu_3per")
    _done = False
    if _resume:
        _st = json.load(open(os.path.join(_resume, "_train_state.json")))
        if int(_st.get("epoch", 0)) >= EPOCHS:
            print(f"✅ model_retrained_iu_3per đã train đủ {EPOCHS} epoch (từ {_resume}) — skip")
            _done = True
        else:
            print(f"♻️  Sẽ resume từ epoch {_st.get('epoch')} ({_resume})")
    if not _done:
        cmd = [
            "python", "scripts/train_iu_model.py",
            "--config", "config_baseline_iu_kaggle.yaml",
            "--mode", "re", "--forget_pct", "3", "--forget_set", IU_FORGET3,
            "--init_from", INIT_DIR,
            "--text_data_dir", IU_META, "--img_data_dir", IU_IMG, "--data_split", IU_SPLIT,
            "--output_dir", RE_OUT,
            "--epochs", str(EPOCHS), "--lr", LR, "--batch_size", BS, "--seed", "42",
        ]
        if _resume:
            cmd += ["--resume_from", _resume]
        print("▶", " ".join(cmd))
        rc = subprocess.run(cmd, env=dict(os.environ, PYTHONPATH=".")).returncode
        if rc != 0:
            raise RuntimeError(f"train model_retrained_iu_3per failed (exit {rc})")


In [ ]:
# ====================================
# CELL 5: Verify output structure → sẵn sàng "Create Dataset from Output"
# ====================================
import os
ROOT = "/kaggle/working/forget-mi-models-iu"
expected = [
    f"{ROOT}/base_model/model_og_IU/pytorch_model.bin",
    f"{ROOT}/base_model/model_og_IU/config.json",
    f"{ROOT}/base_model/model_og_IU/vocab.txt",
    f"{ROOT}/retrained_model/model_retrained_iu_3per/pytorch_model.bin",
    f"{ROOT}/retrained_model/model_retrained_iu_3per/config.json",
    f"{ROOT}/retrained_model/model_retrained_iu_3per/vocab.txt",
]
print("📋 Output forget-mi-models-iu:")
ok = True
for p in expected:
    e = os.path.exists(p)
    ok = ok and e
    size = f"{os.path.getsize(p)/1e6:.1f} MB" if e else ""
    print(f"  {'✅' if e else '❌'} {p}  {size}")

print("\n" + ("="*60))
if ok:
    print("✅ HOÀN TẤT. Bây giờ:")
    print("   1. File → Save Version → Save & Run All (hoặc Quick Save nếu đã chạy xong)")
    print("   2. Sau khi Successful → 'Create Dataset from Output'")
    print("   3. Đặt tên: forget-mi-models-iu  (lowercase, hyphen)")
    print("   4. Dùng trong run_kaggle_baseline.ipynb (Cell 4d) + notebook LoKU")
else:
    print("❌ Thiếu file — kiểm tra Cell 3/Cell 4 đã chạy xong chưa.")

## ✅ Sau khi xong

Cấu trúc `forget-mi-models-iu` (khớp auto-detect của baseline/LoKU):
```
forget-mi-models-iu/
├── base_model/model_og_IU/                  (pytorch_model.bin + config + tokenizer)
└── retrained_model/model_retrained_iu_3per/ (pytorch_model.bin + config + tokenizer)
```

→ Tiếp: `run_kaggle_baseline.ipynb` Cell 4d (`RUN_IU_3PER=True`) và notebook LoKU (`dataset='iu'`, 3%, multi-seed).
